# 05 — Grad-CAM Validation

Generates Grad-CAM activation maps for held-out subjects and verifies the model
is attending to neuroanatomically meaningful regions.

**Expected activations per output:**
| Target | Expected high-activation regions |
|--------|----------------------------------|
| Trait anxiety | Amygdala, anterior insula, dACC |
| Chronic stress | Hypothalamus, hippocampus, vmPFC |
| Neuroticism | mPFC, posterior cingulate, amygdala |
| DMN | mPFC, PCC, angular gyrus |
| Salience | Anterior insula, dACC |
| ECN | dlPFC, posterior parietal |

**Output:** Region score dict (`region_scores.json`) in the same format the FastAPI backend returns.

In [ ]:
!pip install -q tensorflow nibabel nilearn numpy matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, json, os
import tensorflow as tf
from tensorflow import keras

BASE_DIR  = '/content/drive/MyDrive/lemon'
MODEL_DIR = f'{BASE_DIR}/models'

model = keras.models.load_model(f'{MODEL_DIR}/best_model.keras')
print('Model loaded ✓')
model.summary()

In [ ]:
# ── 3D Grad-CAM implementation ────────────────────────────────────────────────

def grad_cam_3d(model, volume, output_name='wellbeing_output', target_idx=0, last_conv='conv3d_2'):
    """
    Compute Grad-CAM for a 3D input volume.

    Parameters
    ----------
    model       : trained Keras model
    volume      : (1, 48, 48, 32, 2) float32 — single subject, channels-last
    output_name : which output head to explain
    target_idx  : which output neuron within that head (0=anxiety, 1=stress, 2=neuro)
    last_conv   : name of the last Conv3D layer to hook gradients on

    Returns
    -------
    cam : (48, 48, 32) float32 — upsampled activation map, normalised 0–1
    """
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv).output,
                 model.get_layer(output_name).output]
    )

    with tf.GradientTape() as tape:
        inp_tensor = tf.cast(volume, tf.float32)
        conv_out, pred = grad_model(inp_tensor)
        target_score   = pred[0, target_idx]

    grads = tape.gradient(target_score, conv_out)[0]  # (h, w, d, C)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))    # (C,) — global avg pool

    # Weighted combination of feature maps
    cam = tf.reduce_sum(conv_out[0] * weights, axis=-1).numpy()  # (h, w, d)
    cam = np.maximum(cam, 0)  # ReLU

    # Upsample to input space (48, 48, 32)
    from scipy.ndimage import zoom
    scale = np.array([48, 48, 32]) / np.array(cam.shape)
    cam   = zoom(cam, scale, order=1)

    # Normalise
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.astype(np.float32)

print('Grad-CAM function defined ✓')

In [ ]:
# ── Named ROIs — MNI coords ───────────────────────────────────────────────────
TARGET_AFFINE = np.diag([4.0, 4.0, 4.5, 1.0])

ROIS = {
    'Amygdala (L)':        (-24, -4, -20),
    'Anterior cingulate':  (0, 28, 24),
    'Insula (R)':          (38, 2, 2),
    'vmPFC':               (0, 52, -8),
    'dlPFC (L)':           (-44, 28, 28),
    'Hippocampus (L)':     (-28, -22, -16),
    'Posterior cingulate': (-2, -52, 26),
    'Angular gyrus (L)':   (-46, -64, 28),
}

def mni_to_vox(mni_coord, affine):
    inv = np.linalg.inv(affine)
    vox = (inv @ np.array([*mni_coord, 1]))[:3]
    return np.round(vox).astype(int)

def score_rois(cam, rois, affine, radius=2):
    """Extract mean CAM value in a small sphere around each ROI centre."""
    scores = {}
    x, y, z = np.mgrid[:48, :48, :32]
    for name, mni in rois.items():
        vox = mni_to_vox(mni, affine)
        vox = np.clip(vox, 0, [47, 47, 31])
        dist = np.sqrt((x-vox[0])**2 + (y-vox[1])**2 + (z-vox[2])**2)
        mask = dist <= radius
        scores[name] = float(cam[mask].mean())
    return scores

print('ROI scoring function defined ✓')

In [ ]:
import matplotlib.pyplot as plt

X     = np.load(f'{BASE_DIR}/X.npy')            # (N, 2, 48, 48, 32)
y_reg = np.load(f'{BASE_DIR}/y_regression.npy') # (N, 3)

# Use last 10% of subjects as a held-out test set
n_test   = max(1, len(X) // 10)
X_test   = np.transpose(X[-n_test:], (0, 2, 3, 4, 1))  # → channels-last
yr_test  = y_reg[-n_test:]

print(f'Test subjects: {n_test}')

TARGET_NAMES = ['trait_anxiety', 'chronic_stress', 'neuroticism']
all_roi_scores = []

for i in range(min(5, n_test)):  # Visualise first 5 test subjects
    vol = X_test[i:i+1]  # (1, 48, 48, 32, 2)

    preds = model.predict(vol, verbose=0)
    wb_pred  = preds[0][0]  # (3,) wellbeing z-scores
    net_pred = preds[1][0]  # (3,) network activations

    print(f'\nSubject {i+1}:')
    print(f'  True  y_reg: {yr_test[i].round(2)}')
    print(f'  Pred  y_reg: {wb_pred.round(2)}')
    print(f'  Net activations (DMN, SN, ECN): {net_pred.round(3)}')

    # Grad-CAM for trait anxiety (target index 0)
    cam = grad_cam_3d(model, vol, output_name='wellbeing_output', target_idx=0)
    roi_scores = score_rois(cam, ROIS, TARGET_AFFINE)
    all_roi_scores.append(roi_scores)
    print(f'  ROI scores: {json.dumps({k: round(v,3) for k,v in roi_scores.items()}, indent=4)}')

    # Visualise middle axial slice
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(vol[0, :, :, 16, 0].T, cmap='gray', origin='lower')
    axes[0].set_title('Mean BOLD — z=16')
    axes[1].imshow(vol[0, :, :, 16, 1].T, cmap='hot', origin='lower')
    axes[1].set_title('fALFF — z=16')
    axes[2].imshow(vol[0, :, :, 16, 0].T, cmap='gray', origin='lower', alpha=0.6)
    axes[2].imshow(cam[:, :, 16].T, cmap='jet', origin='lower', alpha=0.5)
    axes[2].set_title('Grad-CAM (trait anxiety) — z=16')
    plt.tight_layout()
    plt.savefig(f'{BASE_DIR}/gradcam_sub{i+1}.png', dpi=100)
    plt.show()

In [ ]:
# ── Save mean ROI scores across test subjects ─────────────────────────────────
# This is the format the FastAPI backend will return for each scan.
mean_roi = {}
for k in ROIS:
    mean_roi[k] = float(np.mean([s[k] for s in all_roi_scores]))

# Normalise to 0–1
max_score = max(mean_roi.values()) + 1e-8
mean_roi_norm = {k: round(v / max_score, 3) for k, v in mean_roi.items()}

with open(f'{BASE_DIR}/region_scores_example.json', 'w') as f:
    json.dump(mean_roi_norm, f, indent=2)

print('Mean ROI scores (normalised):')
for k, v in sorted(mean_roi_norm.items(), key=lambda x: -x[1]):
    bar = '█' * int(v * 30)
    print(f'  {k:25s} {v:.3f}  {bar}')

In [ ]:
# ── Regression performance on full test set ───────────────────────────────────
from sklearn.metrics import mean_absolute_error, r2_score

preds_all = model.predict(X_test, verbose=0)
wb_preds  = preds_all[0]  # (n_test, 3)

print('\nRegression performance (z-scored targets):')
print(f'{"Target":25s}  MAE    R²')
print('-' * 40)
for i, name in enumerate(TARGET_NAMES):
    mae = mean_absolute_error(yr_test[:, i], wb_preds[:, i])
    r2  = r2_score(yr_test[:, i], wb_preds[:, i])
    print(f'{name:25s}  {mae:.3f}  {r2:.3f}')

print()
print('Interpretation: MAE < 0.8 z-scores = good for n~200 dataset.')
print('R² > 0.2 = model captures real variance (hard benchmark for fMRI prediction).')
print()
print('='*50)
print('ALL NOTEBOOKS COMPLETE')
print('='*50)
print(f'Trained model: {MODEL_DIR}/best_model.keras')
print(f'Wellbeing stats: {BASE_DIR}/wb_stats.json')
print()
print('Next: deploy model behind FastAPI — see the backend stub')
print('in src/api/neuroModel.js for the expected API contract.')